# Intent Graph — MongoDB Atlas (không cần LLM)

Build graph trực tiếp từ CSV đã có hierarchy rõ ràng.
Không cần OpenAI hay bất kỳ LLM nào.

**Schema graph:**
```
domain:Electronics
    └─ l1:after_sale
        └─ l2:refund_request
            └─ l3:smartphone_refund
                └─ intent:INT-7  (Hoàn tiền điện thoại)
```

**2 collections:**
- `intent_nodes` — mỗi node trong graph
- `intent_edges` — directed edges (from → to)

**Yêu cầu:** Thêm `MONGODB_URI` vào Colab Secrets (🔑).

## 1. Cài đặt

## 2. Kết nối MongoDB Atlas

In [35]:
!pip install -q pymongo 'pymongo[srv]'
print('✅ Done')

✅ Done


## 3. Upload CSV

In [36]:
#Option B — mount Google Drive (bỏ comment nếu dùng Drive)
from google.colab import drive
import pathlib
drive.mount('/content/drive')
CSV_PATH = pathlib.Path('/content/drive/MyDrive/intent_kb/data/unified_intents.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
import ssl
import certifi
from google.colab import userdata
from pymongo import MongoClient

# Lấy URI từ Colab Secrets
MONGODB_URI = userdata.get('MONGODB_URI')
assert MONGODB_URI, 'Vui lòng thêm MONGODB_URI vào Colab Secrets (🔑)'

DB_NAME = 'intent_kb'
COL_NODES = 'intent_nodes'
COL_EDGES = 'intent_edges'

print("Đang thử kết nối với cấu hình bỏ qua kiểm tra SSL...")
try:
    # Thêm các tham số tlsAllowInvalidCertificates để debug lỗi SSL handshake
    client = MongoClient(
        MONGODB_URI,
        tls=True,
        tlsCAFile=certifi.where(),
        tlsAllowInvalidCertificates=True,
        serverSelectionTimeoutMS=10000,
        connectTimeoutMS=10000
    )

    # Kiểm tra kết nối
    client.admin.command('ping')
    print("✅ Kết nối thành công tới MongoDB Atlas!")

    db = client[DB_NAME]
    print(f"✅ Database: {db.name}")

    # Kiểm tra nhanh quyền ghi
    res = db.test_connection.insert_one({'status': 'success'})
    db.test_connection.delete_one({'_id': res.inserted_id})
    print("✅ Quyền Đọc/Ghi: OK")

except Exception as e:
    print(f"❌ Vẫn lỗi kết nối: {e}")
    print("\n--- HÀNH ĐỘNG BẮT BUỘC ---")
    print("Lỗi này xác nhận Atlas đang chủ động từ chối kết nối.")
    print("Hãy vào: https://cloud.mongodb.com/")
    print("1. Menu bên trái chọn 'Network Access'.")
    print("2. Nhấn 'Add IP Address'.")
    print("3. Nhấn nút 'ALLOW ACCESS FROM ANYWHERE' (0.0.0.0/0).")
    print("4. Nhấn 'Confirm' và đợi 1 phút rồi chạy lại cell này.")

Đang thử kết nối với cấu hình bỏ qua kiểm tra SSL...
✅ Kết nối thành công tới MongoDB Atlas!
✅ Database: intent_kb
✅ Quyền Đọc/Ghi: OK


## 4. Parse CSV → nodes & edges

In [38]:
import csv

def build_graph(csv_path):
    nodes = {}   # _id -> doc
    edges = set()  # (from_id, to_id)

    def add_node(node_id, level, name, **extra):
        if node_id not in nodes:
            nodes[node_id] = {'_id': node_id, 'level': level, 'name': name, **extra}

    def get_value(row, *keys):
        for key in keys:
            if key in row and row[key] is not None:
                val = row[key].strip()
                if val:
                    return val
        return ''

    add_node('ROOT', 'root', 'Intent Knowledge Base')

    with open(csv_path, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            intent_id = get_value(row, 'STT', 'Ma Intent')
            domain = get_value(row, 'Domain', 'Mien')
            l1 = get_value(row, 'L1 Category', 'Nhom cap 1')
            l2 = get_value(row, 'L2 Intent', 'Intent cap 2')
            l3 = get_value(row, 'L3 Specific Intent', 'Intent cap 3 cu the')

            if not intent_id:
                continue

            domain_id = f'domain:{domain}'
            l1_id = f'l1:{l1}'
            l2_id = f'l2:{l2}'
            l3_id = f'l3:{l3}'

            add_node(domain_id, 'domain', domain)
            add_node(l1_id, 'l1', l1)
            add_node(l2_id, 'l2', l2)
            add_node(l3_id, 'l3', l3)
            add_node(
                intent_id,
                'intent',
                get_value(row, 'Intent Name', 'Ten Intent'),
                description=get_value(row, 'Description', 'Mo ta'),
                confidence=get_value(row, 'Confidence Level', 'Do tin cay'),
                domain=domain,
                product_category=get_value(row, 'Product Category', 'Danh muc san pham'),
                l1=l1,
                l2=l2,
                l3=l3,
                detection_signals=get_value(row, 'Detection Signals', 'Tin hieu nhan dien'),
                example=get_value(row, 'Examples', 'Vi du'),
                category_logic=get_value(row, 'Logic category san pham'),
            )

            # edges: ROOT -> domain -> l1 -> l2 -> l3 -> intent
            for src, dst in [
                ('ROOT', domain_id),
                (domain_id, l1_id),
                (l1_id, l2_id),
                (l2_id, l3_id),
                (l3_id, intent_id),
            ]:
                edges.add((src, dst))

    edge_docs = [
        {'_id': f'{s}->{d}', 'from': s, 'to': d}
        for s, d in edges
    ]
    return list(nodes.values()), edge_docs


nodes, edges = build_graph(CSV_PATH)
print(f'✅ {len(nodes)} nodes  |  {len(edges)} edges')

# Preview
print('\nNode levels:')
from collections import Counter
print(Counter(n['level'] for n in nodes))

✅ 181 nodes  |  182 edges

Node levels:
Counter({'l3': 72, 'intent': 72, 'l2': 32, 'domain': 2, 'l1': 2, 'root': 1})


## 5. Insert vào MongoDB Atlas

In [39]:
# Reset collections
db[COL_NODES].drop()
db[COL_EDGES].drop()

# Insert
db[COL_NODES].insert_many(nodes)
db[COL_EDGES].insert_many(edges)

# Indexes để $graphLookup chạy nhanh
db[COL_NODES].create_index([('level', ASCENDING)])
db[COL_NODES].create_index([('domain', ASCENDING)])
db[COL_EDGES].create_index([('from', ASCENDING)])
db[COL_EDGES].create_index([('to',   ASCENDING)])

print(f'✅ Inserted {db[COL_NODES].count_documents({})} nodes')
print(f'✅ Inserted {db[COL_EDGES].count_documents({})} edges')

✅ Inserted 181 nodes
✅ Inserted 182 edges


In [ ]:
# Helper: upsert nhanh nhãn mới vào graph khi phát sinh từ bước annotation
from datetime import datetime, timezone

def upsert_new_label_to_graph(
    db,
    *,
    domain,
    l1,
    l2,
    l3,
    intent_id,
    intent_name,
    description='',
    confidence='',
    product_category='',
    detection_signals='',
    example='',
    category_logic=''
):
    col_nodes = db[COL_NODES]
    col_edges = db[COL_EDGES]

    domain_id = f'domain:{domain}'
    l1_id = f'l1:{l1}'
    l2_id = f'l2:{l2}'
    l3_id = f'l3:{l3}'

    now = datetime.now(timezone.utc)

    # upsert nodes
    for node_id, level, name, extra in [
        ('ROOT', 'root', 'Intent Knowledge Base', {}),
        (domain_id, 'domain', domain, {}),
        (l1_id, 'l1', l1, {}),
        (l2_id, 'l2', l2, {}),
        (l3_id, 'l3', l3, {}),
        (
            intent_id,
            'intent',
            intent_name,
            {
                'description': description,
                'confidence': confidence,
                'domain': domain,
                'product_category': product_category,
                'l1': l1,
                'l2': l2,
                'l3': l3,
                'detection_signals': detection_signals,
                'example': example,
                'category_logic': category_logic,
            },
        ),
    ]:
        col_nodes.update_one(
            {'_id': node_id},
            {
                '$setOnInsert': {'_id': node_id, 'level': level},
                '$set': {'name': name, **extra, 'updated_at': now},
            },
            upsert=True,
        )

    # upsert edges
    for src, dst in [
        ('ROOT', domain_id),
        (domain_id, l1_id),
        (l1_id, l2_id),
        (l2_id, l3_id),
        (l3_id, intent_id),
    ]:
        edge_id = f'{src}->{dst}'
        col_edges.update_one(
            {'_id': edge_id},
            {
                '$setOnInsert': {'_id': edge_id, 'from': src, 'to': dst},
                '$set': {'updated_at': now},
            },
            upsert=True,
        )

    return {
        'intent_id': intent_id,
        'domain_id': domain_id,
        'l1_id': l1_id,
        'l2_id': l2_id,
        'l3_id': l3_id,
        'status': 'upserted'
    }


# Ví dụ dùng khi có nhãn mới sau annotation:
# result = upsert_new_label_to_graph(
#     db,
#     domain='My pham',
#     l1='truoc_ban',
#     l2='chat_luong_san_pham',
#     l3='ket_cau_san_pham_co_de_bi_moc',
#     intent_id='INT-NEW-001',
#     intent_name='San pham de bi moc lop',
#     description='Hoi ve hien tuong moc lop khi trang diem',
#     confidence='model_generated',
#     product_category='Trang diem',
#     detection_signals='↳ moc lop | ↳ khong an nen',
#     example='Dung co de bi moc khong?',
#     category_logic='Neu co tu khoa moc lop, khong an nen thi gan Trang diem'
# )
# print(result)

In [ ]:
# Guardrail labeling: chỉ cho phép nhãn đã có trong MongoDB graph.
# Nếu phát sinh nhãn mới -> đánh dấu pending review và chỉ upsert khi đủ điều kiện.

from datetime import datetime, timezone


def _collect_allowed_taxonomy(db):
    col_nodes = db[COL_NODES]
    l1_set, l2_set, l3_set = set(), set(), set()

    for doc in col_nodes.find({'level': 'l1'}, {'name': 1}):
        if doc.get('name'):
            l1_set.add(doc['name'])
    for doc in col_nodes.find({'level': 'l2'}, {'name': 1}):
        if doc.get('name'):
            l2_set.add(doc['name'])
    for doc in col_nodes.find({'level': 'l3'}, {'name': 1}):
        if doc.get('name'):
            l3_set.add(doc['name'])

    return {'l1': l1_set, 'l2': l2_set, 'l3': l3_set}


def _is_label_allowed(pred, allowed):
    l1 = pred.get('L1', '').strip()
    l2 = pred.get('L2', '').strip()
    l3 = pred.get('L3', '').strip()
    return (l1 in allowed['l1']) and (l2 in allowed['l2']) and (l3 in allowed['l3'])


def save_annotation_with_guardrails(
    db,
    sample,
    pred,
    *,
    min_conf_auto_approve=0.90,
    min_conf_allow_new_label=0.96,
    allow_auto_add_new_label=False,
    model_name='Qwen2.5-14B-Instruct-GPTQ-Int8',
):
    """
    sample: dict theo schema hasaki_prelabel.json (co sample_id, product_id, sentence, ...)
    pred: {'L1','L2','L3','confidence','reasoning'}
    """
    col_ann = db['intent_annotations']
    now = datetime.now(timezone.utc)

    allowed = _collect_allowed_taxonomy(db)
    confidence = float(pred.get('confidence', 0) or 0)
    valid_existing = _is_label_allowed(pred, allowed)

    # Default status
    qa_status = 'rejected'
    new_label_pending_review = False
    graph_upserted = False

    if valid_existing:
        qa_status = 'approved' if confidence >= min_conf_auto_approve else 'needs_review'
    else:
        # Nhãn chưa có trong taxonomy -> KHÔNG cho pass thẳng
        new_label_pending_review = True
        qa_status = 'pending_new_label_review'

        # Chỉ tự thêm nhãn mới nếu bật cờ + confidence đủ cao
        if allow_auto_add_new_label and confidence >= min_conf_allow_new_label:
            upsert_new_label_to_graph(
                db,
                domain=sample.get('domain') or sample.get('Mien') or 'Unknown',
                l1=pred.get('L1', '').strip(),
                l2=pred.get('L2', '').strip(),
                l3=pred.get('L3', '').strip(),
                intent_id=sample.get('intent_id', f"AUTO-{sample.get('sample_id', 'UNKNOWN')}"),
                intent_name=pred.get('L3', '').strip() or 'new_intent',
                description=pred.get('reasoning', ''),
                confidence=str(confidence),
                product_category=sample.get('category', ''),
                detection_signals=sample.get('sentence', ''),
                example=sample.get('sentence', ''),
                category_logic='auto_added_from_guardrail',
            )
            graph_upserted = True
            qa_status = 'approved_auto_new_label'

    ann_doc = {
        'sample_id': sample.get('sample_id'),
        'product_id': sample.get('product_id'),
        'sentence': sample.get('sentence', ''),
        'category': sample.get('category', ''),
        'model': model_name,
        'intent': {
            'level_1': pred.get('L1', '').strip(),
            'level_2': pred.get('L2', '').strip(),
            'level_3': [pred.get('L3', '').strip()] if pred.get('L3') else [],
        },
        'confidence': confidence,
        'reasoning': pred.get('reasoning', ''),
        'qa_status': qa_status,
        'new_label_pending_review': new_label_pending_review,
        'graph_upserted': graph_upserted,
        'source': sample.get('source', 'hasaki'),
        'updated_at': now,
    }

    col_ann.update_one({'sample_id': ann_doc['sample_id']}, {'$set': ann_doc}, upsert=True)
    return ann_doc


# Example usage:
# sample = {
#   'sample_id': 'abc_q01',
#   'product_id': 'abc',
#   'sentence': 'Dùng có dễ bị mốc không?',
#   'category': 'Trang điểm',
#   'domain': 'My pham',
#   'source': 'hasaki'
# }
# pred = {
#   'L1': 'truoc_ban',
#   'L2': 'chat_luong_san_pham',
#   'L3': 'ket_cau_san_pham_co_de_bi_moc',
#   'confidence': 0.93,
#   'reasoning': 'Khong co ma don, cau hoi truoc mua'
# }
# out = save_annotation_with_guardrails(db, sample, pred, allow_auto_add_new_label=False)
# print(out['qa_status'], out['new_label_pending_review'])

In [ ]:
# Retrieval-first labeling: query MongoDB taxonomy truoc, sau do moi gan nhan
import re


def get_candidate_l3_from_mongodb(db, text, category=None, top_k=10):
    """
    Tra ve danh sach candidate intent (L3 + metadata) tu MongoDB graph.
    Cach don gian: keyword overlap tren node intent (name/description/detection_signals/example).
    """
    tokens = [t.lower() for t in re.findall(r"\w+", text or "") if len(t) >= 2]
    if not tokens:
        return []

    or_conditions = []
    for tok in tokens:
        rgx = {'$regex': re.escape(tok), '$options': 'i'}
        or_conditions.extend([
            {'name': rgx},
            {'description': rgx},
            {'detection_signals': rgx},
            {'example': rgx},
            {'l3': rgx},
        ])

    query = {
        'level': 'intent',
        '$or': or_conditions
    }
    if category:
        query['$or'].append({'product_category': {'$regex': re.escape(category), '$options': 'i'}})

    docs = list(db[COL_NODES].find(
        query,
        {
            '_id': 1, 'name': 1, 'l1': 1, 'l2': 1, 'l3': 1,
            'description': 1, 'detection_signals': 1, 'example': 1,
            'product_category': 1,
        }
    ))

    # score keyword overlap
    scored = []
    token_set = set(tokens)
    for d in docs:
        blob = ' '.join([
            str(d.get('name', '')),
            str(d.get('description', '')),
            str(d.get('detection_signals', '')),
            str(d.get('example', '')),
            str(d.get('l3', '')),
            str(d.get('product_category', '')),
        ]).lower()
        hit = sum(1 for t in token_set if t in blob)
        scored.append((hit, d))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [d for s, d in scored[:top_k] if s > 0]


def build_retrieval_prompt(sample, candidates):
    taxonomy_block = []
    for i, c in enumerate(candidates, 1):
        taxonomy_block.append(
            f"{i}. L1={c.get('l1')} | L2={c.get('l2')} | L3={c.get('l3')} | intent_id={c.get('_id')}"
        )

    taxonomy_text = '\n'.join(taxonomy_block) if taxonomy_block else 'KHONG TIM THAY CANDIDATE'

    prompt = f"""
Ban la bo gan nhan intent cho du lieu ecommerce.
Chi duoc phep chon nhan trong danh sach candidate sau (khong duoc tu y tao nhan moi):
{taxonomy_text}

Input:
- sentence: {sample.get('sentence', '')}
- category: {sample.get('category', '')}
- product_name: {sample.get('product_name', '')}
- brand: {sample.get('brand', '')}

Tra ve JSON:
{{
  "L1": "...",
  "L2": "...",
  "L3": "...",
  "confidence": 0.0,
  "reasoning": "...",
  "is_new_label": false
}}

Neu khong candidate nao phu hop, dat is_new_label=true va confidence<=0.5.
Chi tra ve JSON, khong markdown.
""".strip()
    return prompt


# Example retrieval:
# sample = {
#   'sentence': 'Dung co de bi moc khong?',
#   'category': 'Trang diem',
#   'product_name': 'Bang che khuyet diem',
#   'brand': 'Judydoll'
# }
# cands = get_candidate_l3_from_mongodb(db, sample['sentence'], category=sample['category'], top_k=8)
# print('So candidate:', len(cands))
# for c in cands[:5]:
#     print(c.get('l1'), c.get('l2'), c.get('l3'))
#
# prompt = build_retrieval_prompt(sample, cands)
# print(prompt[:1200])

## 6. Query graph bằng $graphLookup

Traverse toàn bộ cây con từ một node bất kỳ.

In [40]:
def get_subtree(start_id: str, max_depth: int = 5):
    """Trả về tất cả nodes trong cây con bắt đầu từ start_id."""
    pipeline = [
        {'$match': {'_id': start_id}},
        {'$graphLookup': {
            'from':             COL_EDGES,
            'startWith':        '$_id',
            'connectFromField': 'to',
            'connectToField':   'from',
            'as':               'descendant_edges',
            'maxDepth':         max_depth,
        }},
        {'$project': {
            'descendant_ids': '$descendant_edges.to'
        }}
    ]
    result = list(db[COL_NODES].aggregate(pipeline))
    if not result:
        return []
    ids = result[0].get('descendant_ids', [])
    return list(db[COL_NODES].find({'_id': {'$in': ids}}, {'_id': 1, 'level': 1, 'name': 1}))


def find_intents_by(domain=None, l1=None, l2=None):
    """Tìm intent nodes theo filter."""
    q = {'level': 'intent'}
    if domain: q['domain'] = domain
    if l1:     q['l1']     = l1
    if l2:     q['l2']     = l2
    return list(db[COL_NODES].find(q, {'_id': 1, 'name': 1, 'l2': 1, 'l3': 1}))


# ── Demo queries ──────────────────────────────────────────────────────────────

print('=== Subtree của domain Electronics ===')
subtree = get_subtree('domain:Electronics')
for n in subtree:
    print(f"  [{n['level']:8}] {n['_id']}")

print('\n=== Intents thuộc after_sale / refund_request ===')
intents = find_intents_by(l2='refund_request')
for i in intents:
    print(f"  {i['_id']} — {i['name']}  (l3: {i['l3']})")

print('\n=== Tất cả intent Cosmetics after_sale ===')
intents = find_intents_by(domain='Cosmetics', l1='after_sale')
for i in intents:
    print(f"  {i['_id']} — {i['name']}")

=== Subtree của domain Electronics ===
  [intent  ] INT-1
  [intent  ] INT-10
  [intent  ] INT-11
  [intent  ] INT-12
  [intent  ] INT-13
  [intent  ] INT-14
  [intent  ] INT-15
  [intent  ] INT-16
  [intent  ] INT-17
  [intent  ] INT-18
  [intent  ] INT-19
  [intent  ] INT-2
  [intent  ] INT-20
  [intent  ] INT-21
  [intent  ] INT-22
  [intent  ] INT-23
  [intent  ] INT-24
  [intent  ] INT-25
  [intent  ] INT-26
  [intent  ] INT-27
  [intent  ] INT-28
  [intent  ] INT-29
  [intent  ] INT-3
  [intent  ] INT-30
  [intent  ] INT-31
  [intent  ] INT-32
  [intent  ] INT-33
  [intent  ] INT-34
  [intent  ] INT-35
  [intent  ] INT-36
  [intent  ] INT-37
  [intent  ] INT-38
  [intent  ] INT-39
  [intent  ] INT-4
  [intent  ] INT-40
  [intent  ] INT-41
  [intent  ] INT-42
  [intent  ] INT-43
  [intent  ] INT-44
  [intent  ] INT-45
  [intent  ] INT-46
  [intent  ] INT-47
  [intent  ] INT-48
  [intent  ] INT-49
  [intent  ] INT-5
  [intent  ] INT-50
  [intent  ] INT-51
  [intent  ] INT-52
  [int

## 7. Tìm đường đi ngắn nhất giữa 2 nodes

In [41]:
from collections import deque

def find_path(start_id: str, end_id: str) -> list:
    """BFS trên edges để tìm đường từ start → end."""
    edge_map = {}
    for e in db[COL_EDGES].find():
        edge_map.setdefault(e['from'], []).append(e['to'])

    queue   = deque([[start_id]])
    visited = {start_id}
    while queue:
        path = queue.popleft()
        node = path[-1]
        if node == end_id:
            return path
        for nxt in edge_map.get(node, []):
            if nxt not in visited:
                visited.add(nxt)
                queue.append(path + [nxt])
    return []


path = find_path('ROOT', 'INT-7')
print('Path ROOT → INT-7 (Hoàn tiền điện thoại):')
print(' → '.join(path))

Path ROOT → INT-7 (Hoàn tiền điện thoại):
ROOT → domain:Electronics → l1:after_sale → l2:refund_request → l3:smartphone_refund → INT-7


## 8. Xem tổng quan graph

In [42]:
from collections import Counter

level_counts = Counter(
    n['level'] for n in db[COL_NODES].find({}, {'level': 1})
)

print('Graph summary:')
print(f"  {'Level':<12} Count")
print('  ' + '-' * 20)
for level in ['root', 'domain', 'l1', 'l2', 'l3', 'intent']:
    print(f"  {level:<12} {level_counts.get(level, 0)}")
print(f"  {'EDGES':<12} {db[COL_EDGES].count_documents({})}")

Graph summary:
  Level        Count
  --------------------
  root         1
  domain       2
  l1           2
  l2           32
  l3           72
  intent       72
  EDGES        182


## 9. Trực quan hóa Intent Graph (Interactive Visualization)

In [43]:
!pip install -q pyvis

In [44]:
import networkx as nx
from pyvis.network import Network
from IPython.display import HTML

def visualize_graph(limit=200):
    # Khởi tạo NetworkX graph
    G = nx.DiGraph()

    # Lấy nodes và edges từ MongoDB
    nodes_cursor = db[COL_NODES].find().limit(limit)
    edges_cursor = db[COL_EDGES].find().limit(limit)

    # Màu sắc cho các level
    colors = {
        'root': '#FF4B4B',
        'domain': '#FFA500',
        'l1': '#FFFF00',
        'l2': '#00FF00',
        'l3': '#00FFFF',
        'intent': '#FF00FF'
    }

    for node in nodes_cursor:
        G.add_node(
            node['_id'],
            label=node.get('name', node['_id']),
            title=f"Level: {node['level']}",
            color=colors.get(node['level'], '#97C2FC')
        )

    for edge in edges_cursor:
        G.add_edge(edge['from'], edge['to'])

    # Chuyển sang PyVis
    net = Network(height='600px', width='100%', bgcolor='#222222', font_color='white', directed=True, notebook=True, cdn_resources='remote')
    net.from_nx(G)

    # Cấu hình physics để graph tự sắp xếp đẹp hơn
    net.toggle_physics(True)

    # Lưu và hiển thị
    net.show('intent_graph.html')
    return HTML('intent_graph.html')

# Hiển thị graph
visualize_graph()

intent_graph.html


## 10. Tối ưu hóa & Tiện ích bổ sung (Theo Review)

Phần này bổ sung các cải tiến về hiệu suất (Indexes, Batch processing) và các hàm tiện ích để truy vấn graph hiệu quả hơn.

In [45]:
from pymongo import ASCENDING

def setup_indexes():
    """Tạo indexes để tối ưu query."""
    db[COL_NODES].create_index([('level', ASCENDING)])
    db[COL_NODES].create_index([('name', ASCENDING)])
    db[COL_NODES].create_index([('domain', ASCENDING)])

    db[COL_EDGES].create_index([('from', ASCENDING)])
    db[COL_EDGES].create_index([('to', ASCENDING)])
    print("✅ Đã tạo/kiểm tra Indexes cho Nodes và Edges.")

setup_indexes()

✅ Đã tạo/kiểm tra Indexes cho Nodes và Edges.


In [46]:
def get_full_path(intent_id: str):
    """
    Truy vết ngược từ Intent lên tới ROOT để lấy full hierarchy.
    """
    path = []
    current = intent_id

    while current:
        node = db[COL_NODES].find_one({'_id': current}, {'_id': 1, 'name': 1, 'level': 1})
        if node:
            path.insert(0, node)

        # Tìm parent edge
        edge = db[COL_EDGES].find_one({'to': current})
        current = edge['from'] if edge else None

    return path

# Demo
print("=== Full path của INT-7 ===")
full_path = get_full_path('INT-7')
print(' -> '.join([f"{n['name']} ({n['level']})" for n in full_path]))

=== Full path của INT-7 ===
Intent Knowledge Base (root) -> Cosmetics (domain) -> after_sale (l1) -> refund_request (l2) -> smartphone_refund (l3) -> Hoàn tiền điện thoại (intent)


In [34]:
def search_intents(query_text: str):
    """Tìm kiếm intent theo tên hoặc mô tả (Regex)."""
    cursor = db[COL_NODES].find({
        'level': 'intent',
        '$or': [
            {'name': {'$regex': query_text, '$options': 'i'}},
            {'description': {'$regex': query_text, '$options': 'i'}}
        ]
    })
    return list(cursor)

# Demo
results = search_intents("điện thoại")
print(f"=== Kết quả tìm kiếm 'điện thoại' ({len(results)} intents) ===")
for r in results[:5]:
    print(f"- {r['_id']}: {r['name']}")

=== Kết quả tìm kiếm 'điện thoại' (5 intents) ===
- INT-6: Smartphone khác spec
- INT-1: Tư vấn model điện thoại
- INT-5: Smartphone bị hỏng
- INT-7: Hoàn tiền điện thoại
- INT-4: Giao hàng muộn điện thoại
